# Savant — within-target model, `input.csv` only

Two deliberate differences from `train_savant_v2.ipynb`:

**1. `input.csv` alone.** The expanded ChEMBL/BindingDB corpus is not loaded. It contaminated
the v3 benchmark (287/373 ligands over the 0.50 Tanimoto gate before filtering), and after
filtering it bought +0.013 pooled Pearson while *costing* 0.088 within-target r. Freezing the
corpus makes every number here comparable to the published Savant esm8/esm650 columns, which
were also trained on `input.csv`.

**2. Within-target r is the objective, not an afterthought.** Pooled Pearson on the v3
benchmark is a misleading target: its between-target axis is adversarial by construction
(corpus median pKd vs benchmark target mean correlates at **r = −0.17**), so a model is
*penalised* for correctly learning a target's average level. The defensible product is
"rank new chemistry against a target I have data for" — which is exactly within-target r.

So this notebook changes three things to match that claim:

| | v2 notebook | here |
|---|---|---|
| inner validation | protein-disjoint | **scaffold-disjoint, proteins seen** — mirrors deployment |
| early stopping | val RMSE | **val within-target r** (custom metric) |
| models | raw-pKd regressor | raw **+ target-centered residual** regressor |

The target-centered model is the point. Within-target r is invariant to any per-target
constant, so training on `y − mean(y | target)` optimises the metric directly. Note this is
*not* the additive two-part model that already failed (0.474 vs 0.534) — that one predicted
the offset and added it back, and lost because the offset prediction was bad. Here the offset
is only ever taken from training data, and the headline metric never sees it.

Target statistics also enter as **features**, which is what the earlier oracle-offset result
pointed at (oracle offset would have scored 0.614). They use leave-one-out encoding on the
fit rows so the model cannot read a row's own label out of its target's mean.

Everything reads the v2 run's caches as *donors* and writes its own, so the expanded-corpus
embeddings survive intact for the ESMFold2 work.

In [1]:
import gc, json, re, time
from pathlib import Path

import numpy as np
import pandas as pd

HERE   = Path.cwd()                      # run from ML_model_v2/
REPO   = HERE.parent
SRC    = REPO / "input" / "input.csv"
BENCH  = REPO / "ML_model" / "benchmark" / "eval" / "output" / "bench_scored.csv"

## Own workspace. The v2 run's work/ is treated as a read-only DONOR for the expensive
## caches (RDKit descriptors, embeddings) and is never written to - those caches cover the
## expanded corpus and are still needed for the ESMFold2 comparison.
DONOR  = HERE / "work"
OUT    = HERE / "work" / "within"
EMB    = OUT / "emb"
EMB_DONOR = DONOR / "emb"
for d in (OUT, EMB):
    d.mkdir(parents=True, exist_ok=True)

SEED = 1337
MIN_PKD, MAX_PKD = 4.0, 12.0
MAX_LABEL_SD = 1.5
BINDER_PKD = 6.0
CENSOR_CAPS_NM = [10000.0, 100000.0, 1000000.0, 10000000.0]
USE_CAPS_AS_NEGATIVES = True

PROTEIN_ENC = dict(slug="esmc-300m", hf="Synthyra/ESMplusplus_small")
LIGAND_ENC  = dict(slug="chemberta-mlm77", hf="DeepChem/ChemBERTa-77M-MLM")

TEST_PROTEIN_FRAC, TEST_SCAFFOLD_FRAC = 0.20, 0.20
INDOMAIN_FRAC = 0.05
VAL_SCAFFOLD_FRAC = 0.15
MIN_TGT_PAIRS = 10          # a target needs this many pairs to get its own per-target metric
BEDROC_ALPHA = 20.0
N_THREADS = -1
ECFP_BITS = 1024
AA = "ACDEFGHIKLMNPQRSTVWY"

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
log(f"input.csv {SRC}")
log(f"donor caches {DONOR}  ->  writing to {OUT}")

[14:13:08] input.csv /scratch/sguirale/Savant/input/input.csv


[14:13:08] donor caches /scratch/sguirale/Savant/ML_model_v2/work  ->  writing to /scratch/sguirale/Savant/ML_model_v2/work/within


## 1. Load `input.csv`

No expanded corpus. Same cleaning as before: drop censor caps out of the regression target
(keeping them as classifier negatives), clip to a plausible pKd window, collapse replicate
measurements of the same pair to their median.

In [2]:
def load_and_clean(path, source_tag):
    cols = ["seq", "smiles", "neg_log10_affinity_M"]
    have_kd = "affinity_Kd_nM" in pd.read_csv(path, nrows=0).columns
    if have_kd:
        cols.append("affinity_Kd_nM")
    raw = pd.read_csv(path, usecols=cols)
    n0 = len(raw)
    cap = np.zeros(len(raw), bool)
    if have_kd:
        for c in CENSOR_CAPS_NM:
            cap |= np.isclose(raw.affinity_Kd_nM.to_numpy(np.float64), c, rtol=1e-6)
    caps_raw = raw[cap][["seq", "smiles"]].copy()
    meas = raw[~cap]
    meas = meas[meas.neg_log10_affinity_M.between(MIN_PKD, MAX_PKD)].reset_index(drop=True)
    meas["source"] = source_tag
    log(f"  {source_tag}: {n0:,} rows -> {len(meas):,} measurements, "
        f"{int(cap.sum()):,} censor caps")
    return meas, caps_raw


meas_all, caps_all = load_and_clean(SRC, "input.csv")

## Deliberate: the expanded corpus is NOT loaded here. It is not a "missing file" case -
## it exists on disk and is being left out on purpose, so assert that rather than let a
## future reader assume the merge silently failed.
_expanded = HERE / "data" / "expanded_kd_ki.csv"
log(f"expanded corpus deliberately EXCLUDED "
    f"({'present on disk but unused' if _expanded.exists() else 'not present'})")

meas_all["li"] = pd.factorize(meas_all.smiles)[0]
meas_all["pi"] = pd.factorize(meas_all.seq)[0]
ligands  = meas_all.drop_duplicates("li").sort_values("li").smiles.to_numpy()
proteins = meas_all.drop_duplicates("pi").sort_values("pi").seq.to_numpy()
log(f"corpus: {len(meas_all):,} measurement rows, {len(proteins):,} proteins, "
    f"{len(ligands):,} ligands")

lig_of = pd.Series(np.arange(len(ligands)), index=ligands)
pro_of = pd.Series(np.arange(len(proteins)), index=proteins)
caps_all = caps_all.assign(li=lig_of.reindex(caps_all.smiles).to_numpy(),
                           pi=pro_of.reindex(caps_all.seq).to_numpy()).dropna(subset=["li", "pi"])
caps_all["li"] = caps_all.li.astype(int); caps_all["pi"] = caps_all.pi.astype(int)
caps_all = caps_all.drop_duplicates(["li", "pi"])
log(f"censor-cap negatives mapped into the index: {len(caps_all):,}")

[14:13:18]   input.csv: 1,836,729 rows -> 1,689,437 measurements, 82,820 censor caps


[14:13:18] expanded corpus deliberately EXCLUDED (present on disk but unused)


[14:13:19] corpus: 1,689,437 measurement rows, 21,140 proteins, 932,253 ligands


[14:13:19] censor-cap negatives mapped into the index: 51,723


In [3]:
g = meas_all.groupby(["li", "pi"])["neg_log10_affinity_M"]
pairs = g.agg(pkd="median", n_meas="size", label_sd="std").reset_index()
pairs["label_sd"] = pairs.label_sd.fillna(0.0)
dup = pairs[pairs.n_meas > 1]
log(f"unique pairs {len(pairs):,} (collapsed {len(meas_all)-len(pairs):,} rows); "
    f"repeated {len(dup):,} ({100*len(dup)/len(pairs):.1f}%), "
    f"pooled sd {np.sqrt((dup.label_sd**2).mean()):.3f} pKd")
bad = pairs.label_sd > MAX_LABEL_SD
pairs = pairs[~bad].reset_index(drop=True)
pairs["w"] = np.sqrt(pairs.n_meas.to_numpy(np.float32))
caps = caps_all[~caps_all.set_index(["li", "pi"]).index.isin(pairs.set_index(["li", "pi"]).index)]
log(f"final: {len(pairs):,} pairs after dropping {int(bad.sum()):,} irreconcilable "
    f"(label_sd > {MAX_LABEL_SD}); {len(caps):,} capped negatives")
log(f"  binders (pKd >= {BINDER_PKD}): {100*(pairs.pkd >= BINDER_PKD).mean():.1f}%")

## How much of the variance is even within-target? This is the ceiling the whole notebook
## is aiming at - if most variance were between targets, within-target r would be a
## measurement of noise.
_t = pairs.groupby("pi").pkd
_tot = pairs.pkd.var()
_between = _t.mean().reindex(pairs.pi).to_numpy().var()
log(f"variance decomposition: {100*_between/_tot:.1f}% between targets, "
    f"{100*(1-_between/_tot):.1f}% within")

pairs.to_parquet(OUT / "pairs_clean.parquet", index=False)
del meas_all, g; gc.collect()

[14:13:20] unique pairs 1,442,034 (collapsed 247,403 rows); repeated 174,251 (12.1%), pooled sd 0.763 pKd


[14:13:20] final: 1,430,789 pairs after dropping 11,245 irreconcilable (label_sd > 1.5); 43,917 capped negatives


[14:13:20]   binders (pKd >= 6.0): 67.2%


[14:13:20] variance decomposition: 34.9% between targets, 65.1% within


3

## 2. Static blocks — RDKit, ECFP4, AA composition, scaffolds

The v2 run already computed these for 1,168,247 ligands (4h20m of RDKit). `input.csv`'s
ligands are a strict subset, so this **subsets the donor** rather than recomputing. The donor
file is opened read-only and a fresh copy is written under `work/within/`.

In [4]:
from rdkit import Chem, RDLogger
from rdkit.Chem import (Crippen, Descriptors, Lipinski, rdFingerprintGenerator,
                        rdMolDescriptors)
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

LIG_KEYS = ("d11", "dfull", "ecfp", "scaf", "lig_ok")   # rows indexed by ligand
PRO_KEYS = ("aac",)                                     # rows indexed by protein


def _try_donor():
    '''Subset the v2 run's static.npz onto this corpus, or return None.

    Returns None rather than raising for every failure mode: a missing donor, a donor that
    predates saving the ligand/protein order, or a donor that does not cover every ligand
    here. In all three cases the honest answer is "recompute", and the caller does.
    '''
    f = DONOR / "static.npz"
    if not f.exists():
        return None
    z = np.load(f, allow_pickle=True)
    d = {k: z[k] for k in z.files}
    if "ligands" not in d or "proteins" not in d:
        log("  donor static.npz has no ligand/protein order - cannot subset safely")
        return None
    lpos = pd.Series(np.arange(len(d["ligands"])), index=d["ligands"]).reindex(ligands)
    ppos = pd.Series(np.arange(len(d["proteins"])), index=d["proteins"]).reindex(proteins)
    miss_l, miss_p = int(lpos.isna().sum()), int(ppos.isna().sum())
    if miss_l or miss_p:
        log(f"  donor static.npz is missing {miss_l:,} ligands / {miss_p:,} proteins "
            f"- recomputing")
        return None
    li_, pi_ = lpos.to_numpy(int), ppos.to_numpy(int)
    out = {k: d[k][li_] for k in LIG_KEYS}
    out.update({k: d[k][pi_] for k in PRO_KEYS})
    out["ligands"], out["proteins"] = ligands, proteins
    cols = json.loads((DONOR / "rdkit_columns.json").read_text())
    log(f"  SUBSET donor static.npz: {len(d['ligands']):,} -> {len(ligands):,} ligands, "
        f"{len(d['proteins']):,} -> {len(proteins):,} proteins (no RDKit recompute)")
    return out, cols


def build_static():
    f = OUT / "static.npz"
    if f.exists():
        z = np.load(f, allow_pickle=True)
        if (z["ecfp"].shape[0] == len(ligands) and z["aac"].shape[0] == len(proteins)
                and "ligands" in z.files
                and np.array_equal(z["ligands"], ligands)):
            log(f"reusing {f}")
            return {k: z[k] for k in z.files}, json.loads((OUT / "rdkit_columns.json").read_text())
        log(f"{f.name} does not match this corpus - rebuilding")

    donated = _try_donor()
    if donated is not None:
        d, cols = donated
        np.savez_compressed(f, **d)
        (OUT / "rdkit_columns.json").write_text(json.dumps(cols))
        return d, cols

    log("  computing RDKit blocks from scratch (slow - hours)")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ECFP_BITS)
    d11, full, fps, scaf, ok = [], [], [], [], []
    for i, s in enumerate(ligands):
        m = Chem.MolFromSmiles(s); ok.append(m is not None)
        if m is None:
            d11.append([0.0]*11); full.append({}); fps.append(np.zeros(ECFP_BITS)); scaf.append("")
            continue
        mh = Chem.AddHs(m)
        d11.append([Descriptors.MolWt(mh), Crippen.MolLogP(mh), Lipinski.NumHDonors(mh),
                    Lipinski.NumHAcceptors(mh), Lipinski.NumRotatableBonds(mh),
                    rdMolDescriptors.CalcTPSA(mh), rdMolDescriptors.CalcNumRings(mh),
                    rdMolDescriptors.CalcNumAromaticRings(mh), mh.GetNumAtoms(),
                    mh.GetNumHeavyAtoms(), Chem.GetFormalCharge(mh)])
        full.append(Descriptors.CalcMolDescriptors(m))
        fps.append(gen.GetCountFingerprintAsNumPy(m))
        sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m); scaf.append(sc if sc else Chem.MolToSmiles(m))
        if (i+1) % 100_000 == 0: log(f"  ligands {i+1:,}/{len(ligands):,}")
    fd = pd.DataFrame(full).astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    fd = fd.loc[:, fd.max() != fd.min()]
    aac = np.array([[s.count(a)/len(s) for a in AA] + [len(s)] for s in proteins], np.float32)
    d = dict(d11=np.asarray(d11, np.float32), dfull=fd.to_numpy(np.float32),
             ecfp=np.asarray(fps, np.float32), aac=aac, scaf=np.asarray(scaf),
             lig_ok=np.asarray(ok), proteins=proteins, ligands=ligands)
    np.savez_compressed(f, **d)
    (OUT / "rdkit_columns.json").write_text(json.dumps(list(fd.columns)))
    return d, list(fd.columns)


S, RD_COLS = build_static()
pairs = pairs[S["lig_ok"][pairs.li.to_numpy()]].reset_index(drop=True)
caps  = caps[S["lig_ok"][caps.li.to_numpy()]].reset_index(drop=True)
log(f"d11 {S['d11'].shape}  rdkit {S['dfull'].shape}  ecfp {S['ecfp'].shape}  aa {S['aac'].shape}")
log(f"{len(pairs):,} pairs, {len(caps):,} caps after dropping unparseable ligands")

[14:13:35] reusing /scratch/sguirale/Savant/ML_model_v2/work/within/static.npz


[14:13:54] d11 (932253, 11)  rdkit (932253, 215)  ecfp (932253, 1024)  aa (21140, 21)


[14:13:54] 1,430,784 pairs, 43,917 caps after dropping unparseable ligands


## 3. Encoders

Same donor logic: `work/emb/*.npy` from the v2 run covers a superset of these ligands and
proteins, so the caches are subset by key rather than re-embedded. This is what keeps the
whole notebook CPU-only.

In [5]:
def embed_texts(texts, hf, batch_size, max_length):
    import torch
    from transformers import AutoModel, AutoTokenizer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"  {hf} on {dev}, {len(texts):,} items")
    mdl = AutoModel.from_pretrained(hf, trust_remote_code=True).to(dev).eval()
    try:
        tok = AutoTokenizer.from_pretrained(hf, trust_remote_code=True)
    except Exception:
        tok = getattr(mdl, "tokenizer", None)
        if tok is None: raise
    order = np.argsort([len(t) for t in texts]); out = None
    with torch.inference_mode():
        for st in range(0, len(order), batch_size):
            sel = order[st:st+batch_size]
            b = tok([texts[i] for i in sel], padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt")
            b = {k: v.to(dev) for k, v in b.items()}
            h = mdl(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            p = ((h*m).sum(1)/m.sum(1).clamp_min(1)).float().cpu().numpy()
            if out is None: out = np.zeros((len(texts), p.shape[1]), np.float32)
            out[sel] = p
            if (st // batch_size) % 200 == 0:
                log(f"    {min(st+batch_size, len(order)):,}/{len(order):,}")
    del mdl
    if dev == "cuda":
        import torch; torch.cuda.empty_cache()
    return out


def _close(mm_owner):
    '''Release a memmap's file handle.

    np.asarray() on a float32 memmap returns the MEMMAP ITSELF, not a copy, so handing it
    back would keep the donor file open for the life of the notebook. Every path below
    materialises with np.array(...) and then closes.
    '''
    mm = getattr(mm_owner, "_mmap", None)
    if mm is not None:
        mm.close()


def _subset_from(path, keypath, texts, label):
    '''Pull `texts` out of an existing cache by key, or None if it cannot cover them.'''
    if not (path.exists() and keypath.exists()):
        return None
    old = np.load(keypath, allow_pickle=True)
    a = np.load(path, mmap_mode="r")
    if a.shape[0] != len(old):
        log(f"  {label}: cache has {a.shape[0]:,} rows but {len(old):,} keys - ignoring it")
        _close(a); return None
    idx = pd.Series(np.arange(len(old)), index=old).reindex(texts).to_numpy()
    missing = int(np.isnan(idx).sum())
    if missing:
        log(f"  {label}: {missing:,} of {len(texts):,} texts absent - cannot subset")
        _close(a); return None
    out = np.array(a[idx.astype(int)], dtype=np.float32)
    _close(a)
    log(f"  {label}: {len(old):,} -> {len(texts):,} rows by key, no re-embedding")
    return out


def cached(enc, texts, batch_size, max_length):
    '''Own cache -> donor cache -> embed. Writes only under work/within/.'''
    texts = list(texts)
    assert len(set(texts)) == len(texts), f"{enc['slug']}: texts must be unique"
    p, kp = EMB / f"{enc['slug']}.npy", EMB / f"{enc['slug']}_keys.npy"

    arr = _subset_from(p, kp, texts, f"own {enc['slug']}")
    if arr is None:
        arr = _subset_from(EMB_DONOR / f"{enc['slug']}.npy",
                           EMB_DONOR / f"{enc['slug']}_keys.npy",
                           texts, f"donor {enc['slug']}")
    if arr is None:
        log(f"  {enc['slug']}: REBUILDING (this is the step that needs a GPU)")
        arr = embed_texts(texts, enc["hf"], batch_size, max_length)
    np.save(p, arr); np.save(kp, np.asarray(texts, object))
    return arr


P_EMB = cached(PROTEIN_ENC, list(proteins), 8, 1024)
L_EMB = cached(LIGAND_ENC, list(ligands), 512, 256)
log(f"protein {P_EMB.shape}  ligand {L_EMB.shape}")

[14:13:54]   own esmc-300m: 21,140 -> 21,140 rows by key, no re-embedding


[14:13:58]   own chemberta-mlm77: 932,253 -> 932,253 rows by key, no re-embedding


[14:14:01] protein (21140, 960)  ligand (932253, 384)


## 4. Physicochemical complementarity

Unchanged from v2 — 14 features pairing a protein's residue composition against the ligand's
matching property (acidic fraction × ligand positive charge, and so on).

In [6]:
HYDRO = set("AVLIMFWYC")
AROM_P = set("FYWH")
ACID_P = set("DE")
BASIC_P = set("KRH")
HBA_P = set("STYNQDE")
HBD_P = set("STYNQKRHW")

def protein_fracs(seqs):
    n = np.array([len(s) for s in seqs], np.float32)
    def frac(charset):
        return np.array([sum(s.count(c) for c in charset) for s in seqs], np.float32) / n
    return dict(acidic=frac(ACID_P), basic=frac(BASIC_P), hba=frac(HBA_P),
               hbd=frac(HBD_P), hydro=frac(HYDRO), arom=frac(AROM_P),
               log_len=np.log1p(n))

PF = protein_fracs(proteins)

def ligand_scalars(smiles):
    charge, hbd, hba, logp, arom, heavy = [], [], [], [], [], []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        if m is None:
            for lst in (charge, hbd, hba, logp, arom, heavy): lst.append(0.0)
            continue
        charge.append(float(Chem.GetFormalCharge(m)))
        hbd.append(float(Lipinski.NumHDonors(m))); hba.append(float(Lipinski.NumHAcceptors(m)))
        logp.append(float(Crippen.MolLogP(m))); arom.append(float(rdMolDescriptors.CalcNumAromaticRings(m)))
        heavy.append(float(m.GetNumHeavyAtoms()))
    return dict(charge=np.array(charge, np.float32), hbd=np.array(hbd, np.float32),
               hba=np.array(hba, np.float32), logp=np.array(logp, np.float32),
               arom=np.array(arom, np.float32), log_heavy=np.log1p(np.array(heavy, np.float32)))

LS = ligand_scalars(ligands)
log(f"ligand scalar block ready for {len(ligands):,} ligands; "
    f"protein fraction block ready for {len(proteins):,} proteins")


def complementarity_block(li, pi):
    '''(n, 14) float32: 7 raw protein fractions/products, matched to the pair's ligand.'''
    lig_pos = np.maximum(LS["charge"][li], 0); lig_neg = np.maximum(-LS["charge"][li], 0)
    cols = [
        PF["acidic"][pi], lig_pos * PF["acidic"][pi],
        PF["basic"][pi],  lig_neg * PF["basic"][pi],
        PF["hba"][pi],    LS["hbd"][li] * PF["hba"][pi],
        PF["hbd"][pi],    LS["hba"][li] * PF["hbd"][pi],
        PF["hydro"][pi],  LS["logp"][li] * PF["hydro"][pi],
        PF["arom"][pi],   LS["arom"][li] * PF["arom"][pi],
        PF["log_len"][pi], LS["log_heavy"][li] * PF["log_len"][pi],
    ]
    return np.stack(cols, axis=1).astype(np.float32)

COMPLEMENT_NAMES = ["prot_acidic", "charge_pos_x_acidic", "prot_basic", "charge_neg_x_basic",
                    "prot_hba", "lig_hbd_x_prot_hba", "prot_hbd", "lig_hba_x_prot_hbd",
                    "prot_hydro", "lig_logp_x_hydro", "prot_arom", "lig_arom_x_prot_arom",
                    "prot_log_len", "lig_log_heavy_x_log_len"]
_t = complementarity_block(pairs.li.to_numpy()[:100], pairs.pi.to_numpy()[:100])
assert _t.shape == (100, 14) and np.isfinite(_t).all()
log(f"complementarity block: {_t.shape[1]} features per pair, verified finite")

[14:21:45] ligand scalar block ready for 932,253 ligands; protein fraction block ready for 21,140 proteins


[14:21:45] complementarity block: 14 features per pair, verified finite


## 5. Split — outer 2×2, inner validation held out by **scaffold**

The outer 2×2 is unchanged: proteins and Murcko scaffolds are held out independently, so each
cell isolates one kind of novelty.

The inner validation fold changes. v2 held it out **by protein**, which meant every validation
target was unseen — so early stopping was tuned for the novel-protein regime and the target
statistics were always the global fallback. The deployment claim here is new chemistry against
a *known* target, so validation holds out **scaffolds** instead and keeps the proteins seen.

In [7]:
rng = np.random.default_rng(SEED)
pi_all = pairs.pi.to_numpy(); li_all = pairs.li.to_numpy()
y_all  = pairs.pkd.to_numpy(np.float32); w_all = pairs.w.to_numpy(np.float32)
scaf_u = pd.factorize(S["scaf"])[0]; sp_all = scaf_u[li_all]

per_p = pairs.groupby("pi").size(); bins = np.floor(np.log2(per_p.clip(lower=1))).astype(int)
test_p = []
for b in np.unique(bins):
    ps = per_p.index[bins == b].to_numpy()
    test_p.extend(rng.choice(ps, size=max(1, int(round(TEST_PROTEIN_FRAC*len(ps)))),
                             replace=False).tolist())
test_p = set(test_p)
uscaf = np.unique(sp_all)
test_s = set(rng.choice(uscaf, size=max(1, int(round(TEST_SCAFFOLD_FRAC*len(uscaf)))),
                        replace=False).tolist())

npr = np.array([p in test_p for p in pi_all]); ns_ = np.array([s in test_s for s in sp_all])
train_pool = np.where(~npr & ~ns_)[0]; rng.shuffle(train_pool)
n_ind = int(INDOMAIN_FRAC * len(train_pool))
CELLS = {"seen prot / seen chem":   train_pool[:n_ind],
         "seen prot / NOVEL chem":  np.where(~npr & ns_)[0],
         "NOVEL prot / seen chem":  np.where(npr & ~ns_)[0],
         "NOVEL prot / NOVEL chem": np.where(npr & ns_)[0]}
tr_idx = train_pool[n_ind:]
DEPLOY = "seen prot / NOVEL chem"       # the cell this notebook is actually optimising

log(f"train {len(tr_idx):,} pairs / {len(np.unique(pi_all[tr_idx])):,} proteins")
for k, v in CELLS.items():
    mark = "  <- deployment" if k == DEPLOY else ""
    log(f"  {k:24s} {len(v):>9,} pairs / {len(np.unique(pi_all[v])):>6,} proteins{mark}")
tr_prot, tr_scaf = set(pi_all[tr_idx]), set(sp_all[tr_idx])
for k in ("NOVEL prot / seen chem", "NOVEL prot / NOVEL chem"):
    assert not (set(pi_all[CELLS[k]]) & tr_prot), f"protein leak into {k}"
for k in ("seen prot / NOVEL chem", "NOVEL prot / NOVEL chem"):
    assert not (set(sp_all[CELLS[k]]) & tr_scaf), f"scaffold leak into {k}"
log("no protein or scaffold leakage into the cells that claim to hold them out")

## inner validation: scaffolds held out, proteins kept - mirrors the deployment cell
tr_scafs = np.unique(sp_all[tr_idx])
val_s = set(rng.choice(tr_scafs, size=max(1, int(VAL_SCAFFOLD_FRAC*len(tr_scafs))),
                       replace=False).tolist())
in_val = np.array([s in val_s for s in sp_all[tr_idx]])
fit_idx, val_idx = tr_idx[~in_val], tr_idx[in_val]

cap_pi, cap_li = caps.pi.to_numpy(), caps.li.to_numpy(); cap_sp = scaf_u[cap_li]
cap_np_ = np.array([p in test_p for p in cap_pi]); cap_ns = np.array([s in test_s for s in cap_sp])
CAP_TRAIN = np.where(~cap_np_ & ~cap_ns)[0]
cap_in_val = np.array([s in val_s for s in cap_sp[CAP_TRAIN]])
cap_fit, cap_val = CAP_TRAIN[~cap_in_val], CAP_TRAIN[cap_in_val]

_fit_p, _val_p = set(pi_all[fit_idx]), set(pi_all[val_idx])
log(f"fit {len(fit_idx):,} pairs + {len(cap_fit):,} caps | "
    f"val {len(val_idx):,} pairs + {len(cap_val):,} caps")
log(f"  validation proteins also seen in fit: {len(_val_p & _fit_p):,}/{len(_val_p):,} "
    f"({100*len(_val_p & _fit_p)/max(len(_val_p),1):.1f}%) - this is the point of the "
    f"scaffold-held-out fold")
assert not (set(sp_all[val_idx]) & set(sp_all[fit_idx])), "scaffold leak into inner val"

[14:21:47] train 880,819 pairs / 14,961 proteins


[14:21:47]   seen prot / seen chem       46,358 pairs /  3,887 proteins


[14:21:47]   seen prot / NOVEL chem     218,945 pairs /  6,710 proteins  <- deployment


[14:21:47]   NOVEL prot / seen chem     230,041 pairs /  3,822 proteins


[14:21:47]   NOVEL prot / NOVEL chem     54,621 pairs /  1,714 proteins


[14:21:47] no protein or scaffold leakage into the cells that claim to hold them out


[14:21:47] fit 755,137 pairs + 23,246 caps | val 125,682 pairs + 3,427 caps


[14:21:47]   validation proteins also seen in fit: 4,116/5,192 (79.3%) - this is the point of the scaffold-held-out fold


## 6. Target statistics as features (leave-one-out on the fit rows)

The earlier two-part experiment showed an *oracle* per-target offset would score 0.614 against
the joint model's 0.534 — the information is worth a lot, but predicting it additively lost.
So it goes in as a feature instead.

Four features: the target's mean pKd in the fit split, its spread, `log1p(n)` and a seen/unseen
flag. Unseen targets get the global fit mean with `n = 0`, so the model can learn how far to
trust the feature.

**Leave-one-out on fit rows.** If a fit row's target mean included that row's own label, the
model would over-trust the feature — badly, because 1,656 proteins carry 90.8% of the pairs and
the rest have very few. `(sum − yᵢ)/(n − 1)` removes it exactly. Validation, test and benchmark
rows use the full fit mean, which contains none of their labels by construction.

In [8]:
NP = len(proteins)
GLOBAL_MEAN = float(y_all[fit_idx].mean())
GLOBAL_SD = float(y_all[fit_idx].std())

_f = pd.DataFrame({"pi": pi_all[fit_idx], "y": y_all[fit_idx]})
_g = _f.groupby("pi").y.agg(["sum", "mean", "std", "size"])

TGT_MEAN = np.full(NP, GLOBAL_MEAN, np.float32)
TGT_SD   = np.full(NP, GLOBAL_SD,   np.float32)
TGT_N    = np.zeros(NP, np.float32)
TGT_MEAN[_g.index.to_numpy()] = _g["mean"].to_numpy(np.float32)
TGT_SD[_g.index.to_numpy()]   = np.nan_to_num(_g["std"].to_numpy(np.float32), nan=GLOBAL_SD)
TGT_N[_g.index.to_numpy()]    = _g["size"].to_numpy(np.float32)

## Per-PAIR-ROW offset. Fit rows get leave-one-out; everything else gets the full fit mean.
PAIR_OFFSET = TGT_MEAN[pi_all].copy()
_sum = np.zeros(NP, np.float64); _cnt = np.zeros(NP, np.float64)
_sum[_g.index.to_numpy()] = _g["sum"].to_numpy(); _cnt[_g.index.to_numpy()] = _g["size"].to_numpy()
_p, _y = pi_all[fit_idx], y_all[fit_idx].astype(np.float64)
_n = _cnt[_p]
_loo = np.where(_n > 1, (_sum[_p] - _y) / np.maximum(_n - 1, 1), GLOBAL_MEAN)
PAIR_OFFSET[fit_idx] = _loo.astype(np.float32)

log(f"target stats from {len(fit_idx):,} fit rows: {len(_g):,}/{NP:,} proteins seen, "
    f"global mean {GLOBAL_MEAN:.3f} sd {GLOBAL_SD:.3f}")
log(f"  leave-one-out applied to fit rows; mean |LOO - full| = "
    f"{np.abs(PAIR_OFFSET[fit_idx] - TGT_MEAN[pi_all[fit_idx]]).mean():.4f} pKd")

## The LOO offset must not reproduce the row's own label. For a protein with n == 2 the LOO
## mean IS the other row's label, which is correct and expected; what would be wrong is
## LOO == y exactly for larger groups. Check the one thing that would indicate a bug.
_big = fit_idx[_cnt[pi_all[fit_idx]] > 5]
assert not np.allclose(PAIR_OFFSET[_big], y_all[_big]), "LOO offset leaked the row's own label"


def tgt_block(pi, offset):
    '''(n, 4) float32: target mean (LOO where applicable), spread, log1p(n), seen flag.'''
    return np.stack([offset, TGT_SD[pi], np.log1p(TGT_N[pi]),
                     (TGT_N[pi] > 0).astype(np.float32)], axis=1).astype(np.float32)

TGT_NAMES = ["tgt_mean_pkd", "tgt_sd_pkd", "tgt_log_n", "tgt_seen"]
log(f"target-stat block: {len(TGT_NAMES)} features per pair")

[14:21:48] target stats from 755,137 fit rows: 13,885/21,140 proteins seen, global mean 6.666 sd 1.399


[14:21:48]   leave-one-out applied to fit rows; mean |LOO - full| = 0.0184 pKd


[14:21:48] target-stat block: 4 features per pair


## 7. Metrics

One implementation, used for the internal cells, the benchmark, and every reference tool, so
the chapter's tables are mutually comparable.

- **within_r** — Pearson after subtracting each target's mean from both truth and prediction.
  Invariant to any per-target constant, which is exactly why it is the right target here.
- **between_r** — Pearson of the per-target means.
- per-target **Spearman**, **ROC-AUC** and **BEDROC**(α=20) over targets with ≥ 10 pairs.

`EF@1%` is deliberately absent: the v3 benchmark is 88.6% binders, where enrichment saturates
near 1.13 and a perfect ranker scores the same as a mediocre one.

In [9]:
from scipy.stats import spearmanr, t as student_t
from sklearn.metrics import average_precision_score, roc_auc_score
from rdkit.ML.Scoring import Scoring


def _centered(y, p, codes, ng):
    cnt = np.bincount(codes, minlength=ng).astype(np.float64)
    my = np.bincount(codes, weights=y, minlength=ng) / np.maximum(cnt, 1)
    mp = np.bincount(codes, weights=p, minlength=ng) / np.maximum(cnt, 1)
    return y - my[codes], p - mp[codes], my, mp, cnt


def within_r(y, p, g):
    codes, uniq = pd.factorize(g)
    cy, cp, *_ = _centered(np.asarray(y, np.float64), np.asarray(p, np.float64),
                           codes, len(uniq))
    if cy.std() < 1e-9 or cp.std() < 1e-9:
        return np.nan
    return float(np.corrcoef(cy, cp)[0, 1])


def bedroc(y_bool, scores, alpha=BEDROC_ALPHA):
    y_bool = np.asarray(y_bool, bool)
    if y_bool.all() or not y_bool.any():
        return np.nan
    order = np.argsort(-np.asarray(scores, float), kind="stable")
    return float(Scoring.CalcBEDROC([[int(v)] for v in y_bool[order]], 0, alpha))


def metrics(y, p, g, min_pairs=MIN_TGT_PAIRS, absolute=True):
    '''Full metric suite for one (truth, prediction, target) triple.'''
    y = np.asarray(y, np.float64); p = np.asarray(p, np.float64); g = np.asarray(g)
    codes, uniq = pd.factorize(g)
    cy, cp, my, mp, cnt = _centered(y, p, codes, len(uniq))
    row = {"n": len(y), "n_targets": len(uniq)}
    row["within_r"] = (np.nan if cy.std() < 1e-9 or cp.std() < 1e-9
                       else float(np.corrcoef(cy, cp)[0, 1]))
    row["between_r"] = (float(np.corrcoef(my, mp)[0, 1])
                        if len(uniq) > 2 and mp.std() > 1e-12 else np.nan)
    row["pooled_r"] = float(np.corrcoef(y, p)[0, 1]) if p.std() > 1e-12 else np.nan
    row["spearman"] = float(spearmanr(y, p).correlation) if p.std() > 1e-12 else np.nan
    if absolute:
        row["rmse"] = float(np.sqrt(((y - p) ** 2).mean()))
        row["rmse_const"] = float(y.std())
        row["pred_std_ratio"] = float(p.std() / y.std())

    sp, ro, be = [], [], []
    d = pd.DataFrame({"y": y, "p": p, "c": codes})
    for _, sub in d.groupby("c"):
        if len(sub) < min_pairs or sub.p.std() < 1e-12:
            continue
        s = spearmanr(sub.y, sub.p).correlation
        if np.isfinite(s):
            sp.append(s)
        lab = (sub.y.to_numpy() >= BINDER_PKD)
        if lab.any() and not lab.all():
            ro.append(roc_auc_score(lab, sub.p.to_numpy()))
            be.append(bedroc(lab, sub.p.to_numpy()))
    row["n_targets_scored"] = len(sp)
    row["tgt_spearman_median"] = float(np.median(sp)) if sp else np.nan
    row["tgt_spearman_mean"] = float(np.mean(sp)) if sp else np.nan
    row["tgt_spearman_pos_frac"] = float(np.mean(np.array(sp) > 0)) if sp else np.nan
    row["tgt_roc_auc_median"] = float(np.median(ro)) if ro else np.nan
    row["tgt_bedroc_median"] = float(np.nanmedian(be)) if be else np.nan
    return row


def steiger(r12, r13, r23, n):
    '''Steiger (1980) test that two DEPENDENT correlations differ.

    r12 = truth vs model A, r13 = truth vs model B, r23 = model A vs model B, all on the
    same rows. The dependence is the whole point: an unpaired comparison of two r values
    measured against the same truth vector throws away most of the power, which is why the
    earlier +0.082 per-target advantage looked inconclusive.
    '''
    if not all(np.isfinite([r12, r13, r23])) or n < 5:
        return np.nan, np.nan
    R = 1 - r12**2 - r13**2 - r23**2 + 2*r12*r13*r23
    if R <= 0:
        return np.nan, np.nan
    rm = (r12 + r13) / 2
    denom = 2*R*(n - 1)/(n - 3) + rm**2 * (1 - r23)**3
    if denom <= 0:
        return np.nan, np.nan
    tstat = (r12 - r13) * np.sqrt((n - 1) * (1 + r23) / denom)
    return float(tstat), float(2 * student_t.sf(abs(tstat), n - 3))


## --- unit checks, because every one of these has a sign or ordering trap -----------------
_r = np.random.default_rng(0)
_g = np.repeat(np.arange(50), 20)
_off = _r.normal(0, 3, 50)[_g]                    # large per-target offsets
_sig = _r.normal(0, 1, 1000)
_y = _off + _sig
assert abs(within_r(_y, _sig, _g) - 1.0) < 1e-9, "within_r must ignore per-target offsets"
assert abs(within_r(_y, _sig + _off, _g) - 1.0) < 1e-9, "adding the offset must not change it"
## A predictor that is CONSTANT within each target has no within-target variance at all, so
## within_r is undefined rather than zero - centering it leaves exactly 0.0 and the
## correlation has no denominator. This is why the "target mean only" row below reports NaN
## instead of 0: that row exists precisely to show a target-mean predictor carries no
## within-target information, and NaN states that more honestly than a spurious 0.
assert np.isnan(within_r(_y, _off, _g)), "a per-target constant must give undefined within_r"

_lab = np.array([1]*10 + [0]*90, bool)
assert bedroc(_lab, np.arange(100)[::-1]) > 0.99, "perfect ranking -> BEDROC ~ 1"
assert bedroc(_lab, np.arange(100)) < 0.01, "worst ranking -> BEDROC ~ 0"
_t0, _p0 = steiger(0.5, 0.5, 0.7, 500)
assert abs(_t0) < 1e-9 and _p0 > 0.999, "identical correlations must not differ"
_t1, _p1 = steiger(0.45, 0.30, 0.80, 500)
assert _t1 > 0 and _p1 < 0.01, "a 0.15 gap at r23=0.8 on n=500 should be significant"
log("metric unit checks passed (within_r offset-invariance, BEDROC bounds, Steiger)")

[14:21:53] metric unit checks passed (within_r offset-invariance, BEDROC bounds, Steiger)


## 8. Feature assembly and the classifier

The binder/non-binder classifier is kept: per-target ROC-AUC and BEDROC are screening metrics
and the chapter needs them. What is *not* kept is gating the regressor by it — in the v2 run
that cost 0.036 within-target r on the benchmark and 0.08 internally. It is reported below as
a labelled negative result rather than as the headline.

In [10]:
import xgboost as xgb

def assemble(idx, kind="pair"):
    '''(n, width) float32: ligand_emb, d11, rdkit, ecfp, protein_emb, aa, complementarity, tgt.'''
    if kind == "pair":
        li, pi, off = li_all[idx], pi_all[idx], PAIR_OFFSET[idx]
    else:                                    # "cap" - never in the regression fit
        li, pi = cap_li[idx], cap_pi[idx]; off = TGT_MEAN[pi]
    return np.hstack([L_EMB[li], S["d11"][li], S["dfull"][li], S["ecfp"][li],
                      P_EMB[pi], S["aac"][pi], complementarity_block(li, pi),
                      tgt_block(pi, off)]).astype(np.float32)


N_FEATURES = assemble(tr_idx[:5]).shape[1]
log(f"feature width: {N_FEATURES} "
    f"(ligand {L_EMB.shape[1]} + d11 11 + rdkit {S['dfull'].shape[1]} + ecfp {ECFP_BITS} + "
    f"protein {P_EMB.shape[1]} + aa 21 + complementarity 14 + target 4)")

XGB = dict(max_depth=8, eta=0.05, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=10, reg_lambda=2.0, tree_method="hist",
          nthread=N_THREADS, seed=SEED)
TARGET_RECALL = 0.95
N_ROUNDS_REG = 20_000
N_ROUNDS_CLF = 6_000
N_ROUNDS_RANK = 2_000
EARLY_STOP = 200

## N_ROUNDS_CLF was 3,000 in the v2 run and the classifier hit that cap with val-aucpr
## still rising (0.85600 at 2,000 -> 0.86014 at 2,999) - it was round-limited, not
## converged. Doubled; early stopping decides the real endpoint.


def clf_xy(idx, cidx):
    Xs = [assemble(idx, "pair")]
    ys = [(y_all[idx] >= BINDER_PKD).astype(np.int8)]
    ws = [w_all[idx]]
    if USE_CAPS_AS_NEGATIVES and len(cidx):
        Xs.append(assemble(cidx, "cap"))
        ys.append(np.zeros(len(cidx), np.int8))
        ws.append(np.full(len(cidx), float(np.median(w_all[idx])), np.float32))
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)


Xf, yf, wf = clf_xy(fit_idx, cap_fit)
Xv, yv, wv = clf_xy(val_idx, cap_val)
log(f"stage-1 train {Xf.shape}, positive rate {yf.mean():.3f}")
dtr = xgb.QuantileDMatrix(Xf, label=yf, weight=wf); del Xf; gc.collect()
dva = xgb.QuantileDMatrix(Xv, label=yv, ref=dtr)
clf = xgb.train({**XGB, "objective": "binary:logistic", "eval_metric": "aucpr"},
                dtr, num_boost_round=N_ROUNDS_CLF, evals=[(dva, "val")],
                early_stopping_rounds=EARLY_STOP, verbose_eval=250)
pv = clf.inplace_predict(Xv)
THRESH = float(np.quantile(pv[yv == 1], 1 - TARGET_RECALL))
log(f"stage-1 best_iteration {getattr(clf, 'best_iteration', -1)} of {N_ROUNDS_CLF}")
log(f"stage-1 val ROC {roc_auc_score(yv, pv):.4f}  PR {average_precision_score(yv, pv):.4f}  "
    f"threshold@{TARGET_RECALL:.0%} recall = {THRESH:.4f}")
del Xv, dtr, dva; gc.collect()

[14:21:53] feature width: 2633 (ligand 384 + d11 11 + rdkit 215 + ecfp 1024 + protein 960 + aa 21 + complementarity 14 + target 4)


[14:22:02] stage-1 train (778383, 2633), positive rate 0.650


[0]	val-aucpr:0.81668


[250]	val-aucpr:0.92502


[500]	val-aucpr:0.93385


[750]	val-aucpr:0.93827


... (11 outputs left out)

[3666]	val-aucpr:0.94626


[16:27:18] stage-1 best_iteration 3466 of 6000


[16:27:18] stage-1 val ROC 0.9064  PR 0.9463  threshold@95% recall = 0.3476


66

## 9. Regressors — raw pKd vs target-centered residual

Both use identical features and **both early-stop on validation within-target r**, so the
comparison isolates the label transformation and nothing else. XGBoost has no built-in
within-target metric, so it is supplied as a custom one.

`reg_ranking` (`rank:pairwise`, groups = protein) is kept as a negative control. In the v2 run
it reached val-AUC 0.592 and within_r 0.335; it is here to be reported as tried and rejected,
not because it is expected to win.

In [11]:
VAL_CODES, _vu = pd.factorize(pi_all[val_idx])
VAL_NG = len(_vu)
_val_y = y_all[val_idx].astype(np.float64)

def make_within_metric(y_ref):
    '''XGBoost custom metric: within-target Pearson on the validation rows.

    Returned as ("within_r", value) with maximize=True. The label is taken from the caller's
    array rather than the DMatrix, so the same function works for the centered model, whose
    DMatrix label is a residual - centering by target removes any per-target constant, so
    both parameterisations give the identical number.
    '''
    def _m(predt, dmat):
        p = np.asarray(predt, np.float64)
        cy, cp, *_ = _centered(y_ref, p, VAL_CODES, VAL_NG)
        if cy.std() < 1e-9 or cp.std() < 1e-9:
            return "within_r", 0.0
        return "within_r", float(np.corrcoef(cy, cp)[0, 1])
    return _m

WITHIN_METRIC = make_within_metric(_val_y)

def train_reg(label_fit, label_val, tag, rounds=N_ROUNDS_REG):
    Xtr = assemble(fit_idx, "pair")
    dtr = xgb.QuantileDMatrix(Xtr, label=label_fit, weight=w_all[fit_idx])
    del Xtr; gc.collect()
    Xva = assemble(val_idx, "pair")
    dva = xgb.QuantileDMatrix(Xva, label=label_val, ref=dtr); del Xva; gc.collect()
    m = xgb.train({**XGB, "objective": "reg:squarederror"}, dtr,
                  num_boost_round=rounds, evals=[(dva, "val")],
                  custom_metric=WITHIN_METRIC, maximize=True,
                  early_stopping_rounds=EARLY_STOP, verbose_eval=250)
    del dtr, dva; gc.collect()
    log(f"{tag} best_iteration {getattr(m, 'best_iteration', -1)} of {rounds}  "
        f"best val within_r {getattr(m, 'best_score', float('nan')):.4f}")
    return m

reg_squared = train_reg(y_all[fit_idx], _val_y.astype(np.float32), "reg_squared")

[0]	val-rmse:1.37370	val-within_r:0.20988


[250]	val-rmse:0.97564	val-within_r:0.56595


[500]	val-rmse:0.94012	val-within_r:0.60941


[750]	val-rmse:0.91737	val-within_r:0.63447


[1000]	val-rmse:0.90315	val-within_r:0.64956


[1250]	val-rmse:0.89308	val-within_r:0.66020


... (54 outputs left out)

[15000]	val-rmse:0.84034	val-within_r:0.71639


[15250]	val-rmse:0.84028	val-within_r:0.71644


[15485]	val-rmse:0.84027	val-within_r:0.71643


[01:37:43] reg_squared best_iteration 15285 of 20000  best val within_r 0.7164


In [12]:
## The target-centered model. Labels are residuals against the per-target offset - LOO on
## fit rows, full fit mean on validation. Predictions are residuals too; the offset is
## added back only for the absolute-scale metrics, never for within_r.
res_fit = (y_all[fit_idx] - PAIR_OFFSET[fit_idx]).astype(np.float32)
res_val = (y_all[val_idx] - PAIR_OFFSET[val_idx]).astype(np.float32)
log(f"residual labels: fit sd {res_fit.std():.3f} (raw {y_all[fit_idx].std():.3f}), "
    f"val sd {res_val.std():.3f}")
reg_centered = train_reg(res_fit, res_val, "reg_centered")

[01:37:43] residual labels: fit sd 1.135 (raw 1.399), val sd 1.144


[0]	val-rmse:1.14085	val-within_r:0.25307


[250]	val-rmse:0.98102	val-within_r:0.56207


[500]	val-rmse:0.94791	val-within_r:0.60483


[750]	val-rmse:0.92693	val-within_r:0.62909


[1000]	val-rmse:0.91381	val-within_r:0.64358


... (53 outputs left out)

[14500]	val-rmse:0.84911	val-within_r:0.71256


[14750]	val-rmse:0.84913	val-within_r:0.71254


[14760]	val-rmse:0.84913	val-within_r:0.71254


[10:19:12] reg_centered best_iteration 14560 of 20000  best val within_r 0.7126


In [13]:
def sort_by_group(idx):
    return idx[np.argsort(pi_all[idx], kind="stable")]

fit_sorted, val_sorted = sort_by_group(fit_idx), sort_by_group(val_idx)
## qid must be non-decreasing - XGBoost assumes this silently and a violation degrades the
## ranking loss without raising.
assert (np.diff(pi_all[fit_sorted]) >= 0).all(), "fit rows not sorted by group"
assert (np.diff(pi_all[val_sorted]) >= 0).all(), "val rows not sorted by group"

def group_sizes(idx):
    return np.unique(pi_all[idx], return_counts=True)[1]

## No weight= here: once set_group() marks a DMatrix as grouped, XGBoost reinterprets
## `weight` as one value PER GROUP, and a per-row array fails on size mismatch.
Xtr = assemble(fit_sorted, "pair")
dtr = xgb.DMatrix(Xtr, label=y_all[fit_sorted]); del Xtr; gc.collect()
dtr.set_group(group_sizes(fit_sorted))
Xva = assemble(val_sorted, "pair")
dva = xgb.DMatrix(Xva, label=y_all[val_sorted]); del Xva; gc.collect()
dva.set_group(group_sizes(val_sorted))
## eval_metric="ndcg" errors on continuous labels ("must be either 0 or positive integer");
## "auc" is XGBoost's pairwise-concordance metric and accepts real-valued targets.
reg_ranking = xgb.train({**XGB, "objective": "rank:pairwise", "eval_metric": "auc"},
                        dtr, num_boost_round=N_ROUNDS_RANK, evals=[(dva, "val")],
                        early_stopping_rounds=EARLY_STOP, verbose_eval=200)
del dtr, dva; gc.collect()
log(f"reg_ranking best_iteration {getattr(reg_ranking, 'best_iteration', -1)}")

[0]	val-auc:0.54151


[200]	val-auc:0.59150


[400]	val-auc:0.59501


[600]	val-auc:0.60115


[800]	val-auc:0.60452


[990]	val-auc:0.60402


[10:59:45] reg_ranking best_iteration 790


## 10. Internal 2×2 evaluation

Per-pair predictions are written to disk this time, not just the aggregates — without them the
v2 run could not be paired-tested against anything.

In [14]:
rows, preds = [], []
for cell, idx in CELLS.items():
    X = assemble(idx, "pair")
    p_bind = clf.inplace_predict(X)
    p_sq = reg_squared.inplace_predict(X)
    p_ct_res = reg_centered.inplace_predict(X)
    p_rank = reg_ranking.predict(xgb.DMatrix(X))
    del X; gc.collect()
    y, g = y_all[idx], pi_all[idx]
    off = TGT_MEAN[g]                       # never LOO outside the fit split
    p_ct = p_ct_res + off
    gated = np.where(p_bind >= THRESH, p_sq, p_sq.min() - 1.0)

    rows += [
        {"cell": cell, "model": "reg_centered", **metrics(y, p_ct, g)},
        {"cell": cell, "model": "reg_squared", **metrics(y, p_sq, g)},
        {"cell": cell, "model": "reg_ranking", **metrics(y, p_rank, g, absolute=False)},
        {"cell": cell, "model": "reg_squared + gate", **metrics(y, gated, g)},
        {"cell": cell, "model": "target mean only", **metrics(y, off, g)},
    ]
    lab = (y >= BINDER_PKD).astype(int)
    if 0 < lab.sum() < len(lab):
        r = metrics(y, p_bind, g, absolute=False)
        r.update({"cell": cell, "model": "classifier",
                  "roc_auc": roc_auc_score(lab, p_bind),
                  "pr_auc": average_precision_score(lab, p_bind)})
        rows.append(r)
    preds.append(pd.DataFrame({"cell": cell, "pi": g, "li": li_all[idx], "y": y,
                               "p_centered": p_ct, "p_squared": p_sq,
                               "p_ranking": p_rank, "p_binder": p_bind}))
    log(f"{cell}: scored")

R = pd.DataFrame(rows).set_index(["cell", "model"])
R.to_csv(OUT / "internal_2x2.csv")
pd.concat(preds, ignore_index=True).to_parquet(OUT / "internal_2x2_predictions.parquet",
                                               index=False)

KEY = ["within_r", "tgt_spearman_median", "tgt_roc_auc_median", "tgt_bedroc_median",
       "between_r", "pooled_r"]
print("\n=== within-target view (the metric this notebook optimises) ===")
display(R[KEY].round(4))
print("\nwithin_r by cell x model:")
display(R.within_r.unstack("model").reindex(list(CELLS)).round(4))

[11:00:05] seen prot / seen chem: scored


/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation


/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation
/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation


/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation
/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation


/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation


/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation
/tmp/ipykernel_376814/1598258757.py:52: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  s = spearmanr(sub.y, sub.p).correlation


... (31 outputs left out)


=== within-target view (the metric this notebook optimises) ===


within_r  tgt_spearman_median  \
cell                    model                                               
seen prot / seen chem   reg_centered          0.7958               0.7594   
                        reg_squared           0.7989               0.7614   
                        reg_ranking           0.3994               0.3696   
                        reg_squared + gate    0.6863               0.7613   
                        target mean only         NaN                  NaN   
                        classifier            0.6236               0.6210   
seen prot / NOVEL chem  reg_centered          0.7170               0.5706   
                        reg_squared           0.7187               0.5701   
                        reg_ranking           0.3730               0.3416   
                        reg_squared + gate    0.5953               0.5888   
                        target mean only         NaN                  NaN   
                        classifier            0.5586               0.4408   
NOVEL prot / seen chem  reg_centered          0.4591               0.3737   
                        reg_squared           0.4660               0.3803   
                        reg_ranking           0.3077               0.2534   
                        reg_squared + gate    0.3729               0.3835   
                        target mean only         NaN                  NaN   
                        classifier            0.4041               0.3246   
NOVEL prot / NOVEL chem reg_centered          0.3989               0.2952   
                        reg_squared           0.4099               0.2945   
                        reg_ranking           0.2993               0.2217   
                        reg_squared + gate    0.3211               0.2964   
                        target mean only         NaN                  NaN   
                        classifier            0.3544               0.3058   

                                            tgt_roc_auc_median  \
cell                    model                                    
seen prot / seen chem   reg_centered                    0.9184   
                        reg_squared                     0.9167   
                        reg_ranking                     0.7106   
                        reg_squared + gate              0.9044   
                        target mean only                   NaN   
                        classifier                      0.9105   
seen prot / NOVEL chem  reg_centered                    0.8369   
                        reg_squared                     0.8391   
                        reg_ranking                     0.7045   
                        reg_squared + gate              0.8286   
                        target mean only                   NaN   
                        classifier                      0.8316   
NOVEL prot / seen chem  reg_centered                    0.7236   
                        reg_squared                     0.7328   
                        reg_ranking                     0.6431   
                        reg_squared + gate              0.7287   
                        target mean only                   NaN   
                        classifier                      0.7074   
NOVEL prot / NOVEL chem reg_centered                    0.7083   
                        reg_squared                     0.6957   
                        reg_ranking                     0.6420   
                        reg_squared + gate              0.6923   
                        target mean only                   NaN   
                        classifier                      0.6944   

                                            tgt_bedroc_median  between_r  \
cell                    model                                              
seen prot / seen chem   reg_centered                   0.9999     0.8538   
                        reg_squared                    0.9999     0.8551   
                    


within_r by cell x model:


model,classifier,reg_centered,reg_ranking,reg_squared,reg_squared + gate,target mean only
cell,,,,,,
seen prot / seen chem,0.6236,0.7958,0.3994,0.7989,0.6863,NaN
seen prot / NOVEL chem,0.5586,0.7170,0.3730,0.7187,0.5953,NaN
NOVEL prot / seen chem,0.4041,0.4591,0.3077,0.4660,0.3729,NaN
NOVEL prot / NOVEL chem,0.3544,0.3989,0.2993,0.4099,0.3211,NaN


## 11. External benchmark — every tool through one metric implementation

The three Savant models here plus every reference column already in `bench_scored.csv`, scored
identically, then paired Steiger tests of within-target r against each reference.

Two things to keep in mind when reading the absolute-scale columns:

- The benchmark's between-target axis is adversarial (corpus median vs benchmark target mean,
  r = −0.17), so `between_r`, `pooled_r` and `rmse` are expected to look *worse* for a model
  that uses target offsets honestly. That is the known property of this benchmark, not a
  regression.
- `constant_pk` has zero variance, so its correlations are undefined rather than zero.

In [15]:
bm = pd.read_csv(BENCH).dropna(subset=["target_sequence", "ligand_smiles",
                                       "neg_log10_affinity_M"]).reset_index(drop=True)
keep = [Chem.MolFromSmiles(s) is not None for s in bm.ligand_smiles]
bm = bm[keep].reset_index(drop=True)
log(f"benchmark: {len(bm)} pairs, {bm.uniprot_id.nunique()} targets")

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ECFP_BITS)
d11b, fullb, fpb = [], [], []
for s in bm.ligand_smiles:
    m = Chem.MolFromSmiles(s); mh = Chem.AddHs(m)
    d11b.append([Descriptors.MolWt(mh), Crippen.MolLogP(mh), Lipinski.NumHDonors(mh),
                 Lipinski.NumHAcceptors(mh), Lipinski.NumRotatableBonds(mh),
                 rdMolDescriptors.CalcTPSA(mh), rdMolDescriptors.CalcNumRings(mh),
                 rdMolDescriptors.CalcNumAromaticRings(mh), mh.GetNumAtoms(),
                 mh.GetNumHeavyAtoms(), Chem.GetFormalCharge(mh)])
    fullb.append(Descriptors.CalcMolDescriptors(m))
    fpb.append(gen.GetCountFingerprintAsNumPy(m))
fdb = (pd.DataFrame(fullb).astype(np.float32).replace([np.inf, -np.inf], np.nan)
         .fillna(0.0).reindex(columns=RD_COLS, fill_value=0.0).to_numpy(np.float32))

Lb = cached(dict(slug="bench_ligand_" + LIGAND_ENC["slug"], hf=LIGAND_ENC["hf"]),
            list(dict.fromkeys(bm.ligand_smiles)), 256, 256)
Lb = pd.DataFrame(Lb, index=list(dict.fromkeys(bm.ligand_smiles))).reindex(
        bm.ligand_smiles).to_numpy(np.float32)
Pb = cached(dict(slug="bench_protein_" + PROTEIN_ENC["slug"], hf=PROTEIN_ENC["hf"]),
            list(dict.fromkeys(bm.target_sequence)), 8, 1024)
Pb = pd.DataFrame(Pb, index=list(dict.fromkeys(bm.target_sequence))).reindex(
        bm.target_sequence).to_numpy(np.float32)

aab = np.array([[s.count(a)/len(s) for a in AA] + [len(s)] for s in bm.target_sequence], np.float32)
bpf = protein_fracs(list(bm.target_sequence)); bls = ligand_scalars(list(bm.ligand_smiles))
bcomp = np.stack([
    bpf["acidic"], np.maximum(bls["charge"], 0)*bpf["acidic"],
    bpf["basic"],  np.maximum(-bls["charge"], 0)*bpf["basic"],
    bpf["hba"],    bls["hbd"]*bpf["hba"],
    bpf["hbd"],    bls["hba"]*bpf["hbd"],
    bpf["hydro"],  bls["logp"]*bpf["hydro"],
    bpf["arom"],   bls["arom"]*bpf["arom"],
    bpf["log_len"], bls["log_heavy"]*bpf["log_len"],
], axis=1).astype(np.float32)

## Target statistics for benchmark rows come from the TRAINING corpus by sequence. A
## benchmark target with no training data falls back to the global mean with n = 0, the
## same as any unseen protein - so this reports how many of the 20 are actually "seen".
b_pi = pro_of.reindex(bm.target_sequence).to_numpy()
b_seen = ~pd.isna(b_pi)
b_off = np.where(b_seen, TGT_MEAN[np.nan_to_num(b_pi, nan=0).astype(int)], GLOBAL_MEAN)
b_sd = np.where(b_seen, TGT_SD[np.nan_to_num(b_pi, nan=0).astype(int)], GLOBAL_SD)
b_n = np.where(b_seen, TGT_N[np.nan_to_num(b_pi, nan=0).astype(int)], 0.0)
btgt = np.stack([b_off, b_sd, np.log1p(b_n), (b_n > 0).astype(np.float32)],
                axis=1).astype(np.float32)
log(f"benchmark targets with training data: "
    f"{bm.loc[b_seen, 'uniprot_id'].nunique()}/{bm.uniprot_id.nunique()} "
    f"({int(b_seen.sum())}/{len(bm)} pairs)")

Xb = np.hstack([Lb, np.asarray(d11b, np.float32), fdb, np.asarray(fpb, np.float32),
                Pb, aab, bcomp, btgt]).astype(np.float32)
assert Xb.shape[1] == N_FEATURES, f"benchmark width {Xb.shape[1]} != train {N_FEATURES}"
log(f"benchmark matrix {Xb.shape}")

[11:02:30] benchmark: 500 pairs, 20 targets


[11:02:36]   bench_ligand_chemberta-mlm77: REBUILDING (this is the step that needs a GPU)


[11:03:15]   DeepChem/ChemBERTa-77M-MLM on cuda, 373 items


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[11:03:19]     256/373


... (3 outputs left out)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


[11:03:39]     8/20


[11:03:40] benchmark targets with training data: 19/20 (490/500 pairs)


[11:03:40] benchmark matrix (500, 2633)


In [16]:
yb = bm.neg_log10_affinity_M.to_numpy(np.float64)
gb = bm.uniprot_id.to_numpy()

p_sq_b = reg_squared.inplace_predict(Xb)
p_ct_b = reg_centered.inplace_predict(Xb) + b_off
p_rk_b = reg_ranking.predict(xgb.DMatrix(Xb))
p_bd_b = clf.inplace_predict(Xb)
gated_b = np.where(p_bd_b >= THRESH, p_sq_b, p_sq_b.min() - 1.0)

SAVANT = {"Savant within (centered)": p_ct_b,
          "Savant within (squared)": p_sq_b,
          "Savant within (ranking)": p_rk_b,
          "Savant within (squared + gate)": gated_b,
          "Savant within (classifier score)": p_bd_b}

REF = {"Boltz-2": "boltz_affinity_pic50", "PRODIGY-LIG": "prodigy_pkd",
       "Savant esm8": "savant_esm8_pk", "Savant esm650": "savant_esm650_pk",
       "Savant-v2 esm8": "savant_v2_esm8_pk", "Savant-v2 esm650": "savant_v2_esm650_pk",
       "PLAPT": "plapt_pk", "QSAR per-target": "qsar_pk", "Tanimoto kNN": "knn_pk",
       "Constant mean": "constant_pk"}

brows = []
for name, p in SAVANT.items():
    brows.append({"tool": name, **metrics(yb, p, gb,
                                          absolute="ranking" not in name and "classifier" not in name)})
for name, col in REF.items():
    if col not in bm.columns:
        log(f"  {name}: column {col} absent - skipped"); continue
    ok = np.isfinite(bm[col].to_numpy(np.float64))
    if ok.sum() < 10:
        log(f"  {name}: only {int(ok.sum())} finite predictions - skipped"); continue
    brows.append({"tool": name, **metrics(yb[ok], bm[col].to_numpy(np.float64)[ok], gb[ok])})

B = pd.DataFrame(brows).set_index("tool").sort_values("within_r", ascending=False)
B.to_csv(OUT / "benchmark_metrics.csv")

bp = pd.DataFrame({"uniprot_id": gb, "ligand_smiles": bm.ligand_smiles, "y": yb,
                   **{k: v for k, v in SAVANT.items()}})
for name, col in REF.items():
    if col in bm.columns:
        bp[name] = bm[col].to_numpy()
bp.to_parquet(OUT / "benchmark_predictions.parquet", index=False)

print("\n=== v3 benchmark, all tools, one metric implementation ===")
display(B[["n", "n_targets_scored", "within_r", "tgt_spearman_median",
           "tgt_roc_auc_median", "tgt_bedroc_median", "between_r", "pooled_r"]].round(4))
log(f"per-pair predictions -> {OUT/'benchmark_predictions.parquet'}")


=== v3 benchmark, all tools, one metric implementation ===


,n,n_targets_scored,within_r,tgt_spearman_median,tgt_roc_auc_median,tgt_bedroc_median,between_r,pooled_r
tool,,,,,,,,
Savant esm650,500,20,0.4358,0.2776,0.7857,0.9993,0.1003,0.2275
Savant esm8,500,20,0.4327,0.1689,0.7500,0.9996,0.2565,0.3304
Savant-v2 esm650,500,20,0.4286,0.2879,0.7222,0.9999,0.1188,0.2291
Savant-v2 esm8,500,20,0.4045,0.2109,0.8182,1.0000,0.1899,0.2899
Boltz-2,496,20,0.3859,0.2418,0.7024,0.9998,0.3505,0.4120
Savant within (centered),500,20,0.3469,0.2848,0.7407,0.9975,-0.1275,0.0221
Savant within (classifier score),500,20,0.3398,0.1796,0.6667,0.9975,0.0021,0.0780
Savant within (squared),500,20,0.3315,0.1543,0.5294,0.9114,-0.1625,-0.0007
Savant within (squared + gate),500,20,0.3214,0.1021,0.5556,0.9114,-0.0037,0.0791


[11:03:41] per-pair predictions -> /scratch/sguirale/Savant/ML_model_v2/work/within/benchmark_predictions.parquet


In [17]:
## Paired comparison of within-target r. Steiger's test needs the three correlations on the
## SAME rows, so each reference is compared on the intersection where both are finite, and
## n is the within-target degrees of freedom (rows minus targets), not the row count.
HEAD = "Savant within (centered)"
head_p = SAVANT[HEAD]

srows = []
for name, col in REF.items():
    if col not in bm.columns:
        continue
    ref = bm[col].to_numpy(np.float64)
    ok = np.isfinite(ref) & np.isfinite(head_p)
    if ok.sum() < 20 or np.nanstd(ref[ok]) < 1e-12:
        continue
    codes, uniq = pd.factorize(gb[ok])
    cy, ch, *_ = _centered(yb[ok], np.asarray(head_p[ok], np.float64), codes, len(uniq))
    _, cr, *_ = _centered(yb[ok], ref[ok], codes, len(uniq))
    if cr.std() < 1e-12:
        continue
    r12 = float(np.corrcoef(cy, ch)[0, 1])
    r13 = float(np.corrcoef(cy, cr)[0, 1])
    r23 = float(np.corrcoef(ch, cr)[0, 1])
    tstat, pval = steiger(r12, r13, r23, int(ok.sum()) - len(uniq))
    srows.append({"vs": name, "n": int(ok.sum()), "df": int(ok.sum()) - len(uniq),
                  "within_r_savant": r12, "within_r_ref": r13, "delta": r12 - r13,
                  "model_agreement_r": r23, "steiger_t": tstat, "p_value": pval})

Sg = pd.DataFrame(srows).sort_values("delta", ascending=False)
Sg.to_csv(OUT / "benchmark_steiger.csv", index=False)
print(f"\n=== paired Steiger test: {HEAD} vs each reference, within-target r ===")
print("positive delta = Savant higher; p < 0.05 = the difference is real, not sampling noise\n")
display(Sg.round(4))


=== paired Steiger test: Savant within (centered) vs each reference, within-target r ===
positive delta = Savant higher; p < 0.05 = the difference is real, not sampling noise



,vs,n,df,within_r_savant,within_r_ref,delta,model_agreement_r,steiger_t,p_value
8,Tanimoto kNN,458,440,0.3705,0.0664,0.3040,0.1776,5.2931,0.0000
7,QSAR per-target,458,440,0.3705,0.1554,0.2151,0.4065,4.4214,0.0000
6,PLAPT,500,480,0.3469,0.2084,0.1385,0.2995,2.7211,0.0067
1,PRODIGY-LIG,496,476,0.3345,0.2369,0.0976,0.2342,1.8269,0.0683
0,Boltz-2,496,476,0.3345,0.3859,-0.0514,0.5158,-1.2425,0.2147
4,Savant-v2 esm8,500,480,0.3469,0.4045,-0.0576,0.6328,-1.6137,0.1072
5,Savant-v2 esm650,500,480,0.3469,0.4286,-0.0817,0.6757,-2.4533,0.0145
2,Savant esm8,500,480,0.3469,0.4327,-0.0858,0.6613,-2.5268,0.0118
3,Savant esm650,500,480,0.3469,0.4358,-0.0888,0.6452,-2.5619,0.0107


## 12. Save

In [18]:
import joblib

bundle = {
    "variant": "savant_within_input_only",
    "protein_encoder": PROTEIN_ENC["hf"], "ligand_encoder": LIGAND_ENC["hf"],
    "block_order": ["ligand_emb", "d11", "rdkit_named", "ecfp", "protein_emb", "aa",
                    "complementarity", "target_stats"],
    "complementarity_names": COMPLEMENT_NAMES, "target_stat_names": TGT_NAMES,
    "rdkit_columns": RD_COLS, "ecfp_bits": ECFP_BITS,
    "classifier_json": clf.save_raw(raw_format="json"),
    "regressor_squared_json": reg_squared.save_raw(raw_format="json"),
    "regressor_centered_json": reg_centered.save_raw(raw_format="json"),
    "regressor_ranking_json": reg_ranking.save_raw(raw_format="json"),
    "binder_pkd": BINDER_PKD, "threshold": THRESH, "target_recall": TARGET_RECALL,
    "caps_as_negatives": USE_CAPS_AS_NEGATIVES,
    ## The centered model predicts a RESIDUAL. Inference must add the target's training
    ## mean back; without these it is unusable, so they ship with the model.
    "target_offsets": {"proteins": proteins, "mean": TGT_MEAN, "sd": TGT_SD, "n": TGT_N,
                       "global_mean": GLOBAL_MEAN, "global_sd": GLOBAL_SD},
    "data_sources": {"input_csv": str(SRC), "expanded_csv": None},
    "split": {"mode": "protein+scaffold double-disjoint; inner val by scaffold",
              "seed": SEED, "test_protein_frac": TEST_PROTEIN_FRAC,
              "test_scaffold_frac": TEST_SCAFFOLD_FRAC,
              "val_scaffold_frac": VAL_SCAFFOLD_FRAC,
              "cell_sizes": {k: int(len(v)) for k, v in CELLS.items()},
              "deployment_cell": DEPLOY},
    "early_stopping_metric": "within-target Pearson r on the scaffold-held-out inner fold",
    "n_train_pairs": int(len(fit_idx)), "n_train_caps": int(len(cap_fit)),
    "internal_2x2": R.reset_index().to_dict(orient="records"),
    "benchmark": B.reset_index().to_dict(orient="records"),
    "benchmark_steiger": Sg.to_dict(orient="records"),
    "xgboost_version": xgb.__version__,
}
p = OUT / "savant_within_bundle.pkl"
joblib.dump(bundle, p, compress=3)
print(f"{p.name}  {p.stat().st_size/1e6:.0f} MB")

print("\nartifacts for the thesis chapter:")
for f in ["internal_2x2.csv", "internal_2x2_predictions.parquet", "benchmark_metrics.csv",
          "benchmark_predictions.parquet", "benchmark_steiger.csv"]:
    q = OUT / f
    print(f"  {'OK ' if q.exists() else 'MISSING'} {q}")

savant_within_bundle.pkl  237 MB

artifacts for the thesis chapter:
  OK  /scratch/sguirale/Savant/ML_model_v2/work/within/internal_2x2.csv
  OK  /scratch/sguirale/Savant/ML_model_v2/work/within/internal_2x2_predictions.parquet
  OK  /scratch/sguirale/Savant/ML_model_v2/work/within/benchmark_metrics.csv
  OK  /scratch/sguirale/Savant/ML_model_v2/work/within/benchmark_predictions.parquet
  OK  /scratch/sguirale/Savant/ML_model_v2/work/within/benchmark_steiger.csv
